# Chapter 1: Common Conventions and API Elements of scikit-learn

## Comprehensive Summary
This chapter establishes the foundational architecture and design philosophy of scikit-learn, which dictates how all subsequent machine learning workflows are constructed. It explores the strictly enforced, uniform Object-Oriented Programming (OOP) interfaces—Estimators, Transformers, and Pipelines—that allow practitioners to seamlessly swap algorithms, automate preprocessing, and integrate custom components without altering the overall code structure.

## In-Depth Theoretical Analysis & Mechanism Breakdown

### 1. The Design Philosophy
scikit-learn is built upon four core principles: **Consistency** (uniform interfaces), **Simplicity** (sensible default hyperparameters), **Modularity** (independent, chainable building blocks), and **Reusability** (components seamlessly transfer across projects)[cite: 5]. This strict API contract ensures that transitioning from a simple linear model to a complex ensemble requires minimal code refactoring[cite: 5].

### 2. Estimators: The Core Abstraction
An estimator is any object capable of learning parameters from data via the `fit()` method[cite: 5].

*   **Supervised Learning:** The estimator learns a mapping $\hat{f}: \mathbf{X} \rightarrow \mathbf{y}$[cite: 5].
    *   *Example (Linear Regression):* The algorithm minimizes the sum of squared residuals: $\min_{\mathbf{w}, b} \sum_{i=1}^{n} \left( y_i - (\mathbf{w}^T \mathbf{x}_i + b) \right)^2$[cite: 5]. Behind the scenes of `.fit()`, it solves the normal equations $\mathbf{w} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}$ to find the optimal weight vector $\mathbf{w}$ and bias $b$[cite: 5].
*   **Unsupervised Learning:** Uses `fit_predict()` to simultaneously learn from data and assign labels to it, as there is no separate holdout set during training[cite: 5].
    *   *Example (K-Means):* Minimizes the within-cluster sum of squares: $\min_{\{C_k\}_{k=1}^{K}} \sum_{k=1}^{K} \sum_{\mathbf{x}_i \in C_k} \Vert{}\mathbf{x}_i - \boldsymbol{\mu}_k\Vert{}^2$, where $\boldsymbol{\mu}_k$ is the cluster centroid[cite: 5].

### 3. Transformers and Data Leakage Prevention
Transformers are estimators designed to preprocess data rather than make predictions[cite: 5].
*   **Mathematical Operation (StandardScaler):** Centers each feature at zero mean and unit variance using the formula $z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$[cite: 5].
*   **API Discipline:** The library enforces using `fit_transform()` strictly on training data (to learn $\mu$ and $\sigma$), and only `transform()` on test data[cite: 5]. Re-fitting on test data overwrites the learned parameters, leading to critical data leakage bugs[cite: 5].

### 4. Custom Estimators and Mixin Composition
scikit-learn's API is fully extensible via subclassing `BaseEstimator` alongside specific Mixin classes (`ClassifierMixin`, `RegressorMixin`, `TransformerMixin`)[cite: 5].
*   Mixins are a Python OOP pattern that injects default behaviors (like the `fit_transform()` or `score()` methods) without creating deep, complex inheritance trees[cite: 5].
*   Custom estimators must isolate hyperparameter assignment strictly within the `__init__` constructor and use `check_is_fitted()` to validate the estimator state before transformation[cite: 5].

### 5. Pipelines and MLOps Integration
A `Pipeline` sequentially chains $K$ transformers and a final predictive estimator into a single, deployable artifact[cite: 5].
*   **Mathematical Sequence:** $\hat{y} = E\left(T_K\left(\ldots T_2\left(T_1(\mathbf{X})\right)\right)\right)$[cite: 5].
*   Pipelines guarantee that identical preprocessing logic is applied to both training and production data, forming the backbone of automated Continuous Training (CT) and CI/CD workflows in MLOps architectures[cite: 5].

### 6. Evaluation Metrics and Hyperparameter Search
*   **Model Attributes:** Learned parameters are distinguished by a trailing underscore (e.g., `coef_`, `intercept_`)[cite: 5].
*   **Metric Limitations ($R^2$):** While regressors default to the $R^2$ score, this metric artificially inflates as more features are added[cite: 5]. The notebook highlights the necessity of the **Adjusted $R^2$**: $R^2_{\text{adj}} = 1 - \frac{(1 - R^2)(n - 1)}{n - p - 1}$, which mathematically penalizes unnecessary model complexity[cite: 5].
*   **Hyperparameters & Impurity:** Methods like `set_params()` and `get_params()` expose configurations to search grids. For example, Tree-based estimators dictate split quality via Gini Impurity: $G = 1 - \sum_{c=1}^{C} p_c^2$, driving the model's structural decisions before training begins[cite: 5].

### 7. Metadata Routing and Estimator Tags
Modern scikit-learn features include metadata routing and `__sklearn_tags__()`, which explicitly define an estimator's capabilities (e.g., multiclass support, required inputs)[cite: 5]. This allows grid searches and pipelines to intelligently dynamically route auxiliary data, such as `sample_weight`, exclusively to the components that consume it[cite: 5].

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Verify scikit-learn version
import sklearn
print(f"scikit-learn version: {sklearn.__version__}")
print(f"NumPy version: {np.__version__}")

scikit-learn version: 1.6.1
NumPy version: 2.0.2


In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Example data
X = np.array([[1], [2], [3], [4], [5]])  # Feature matrix (5 samples, 1 feature)
y = np.array([1, 2, 3, 3.5, 5])          # Target values

# Create and fit the model
model = LinearRegression()
model.fit(X, y)

# Predict values for new data
X_new = np.array([[6], [7]])
predictions = model.predict(X_new)
print("Predictions for X = [6, 7]:", predictions)

Predictions for X = [6, 7]: [5.75 6.7 ]


In [ ]:
from sklearn.cluster import KMeans
import numpy as np

# Example data -- 5 points in 1D
X = np.array([[1], [2], [3], [4], [5]])

# KMeans clustering with 2 clusters
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)
print("Cluster labels:", labels)
print("Cluster centers:", kmeans.cluster_centers_.flatten())

Cluster labels: [0 0 0 1 1]
Cluster centers: [2.  4.5]


In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Example data: 3 samples, 2 features
X = np.array([[1, 2], [3, 4], [5, 6]])

# Create a StandardScaler instance
scaler = StandardScaler()

# Fit the scaler on the data (learns mean and std)
scaler.fit(X)

# Transform the data
X_scaled = scaler.transform(X)
print("Original data:")
print(X)
print()
print("Scaled data:")
print(X_scaled)
print()
print("Mean of each feature (learned):", scaler.mean_)
print("Std of each feature (learned):", scaler.scale_)

Original data:
[[1 2]
 [3 4]
 [5 6]]

Scaled data:
[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Mean of each feature (learned): [3. 4.]
Std of each feature (learned): [1.63299316 1.63299316]


In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Example data
X = np.array([[1, 2], [3, 4], [5, 6]])

# Create a StandardScaler instance and fit_transform in one step
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Scaled data (via fit_transform):")
print(X_scaled)

# Verify: mean ≈ 0, std = 1 for each feature
print()
print("Column means after scaling:", X_scaled.mean(axis=0))
print("Column stds after scaling:", X_scaled.std(axis=0))

Scaled data (via fit_transform):
[[-1.22474487 -1.22474487]
 [ 0.          0.        ]
 [ 1.22474487  1.22474487]]

Column means after scaling: [0. 0.]
Column stds after scaling: [1. 1.]


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
import numpy as np

class ValueClipper(BaseEstimator, TransformerMixin):
    """Clips all feature values to [lower, upper] range."""

    def __init__(self, lower=0.0, upper=1.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        # No parameters to learn, but we mark the estimator as fitted
        self.n_features_in_ = X.shape[1] if hasattr(X, 'shape') else len(X[0])
        self.is_fitted_ = True
        return self

    def transform(self, X):
        check_is_fitted(self, 'is_fitted_')
        X = np.array(X)
        return np.clip(X, self.lower, self.upper)

# Demonstrate usage
X = np.array([[0.5, -0.3], [1.5, 0.7], [0.2, 2.1]])
clipper = ValueClipper(lower=0.0, upper=1.0)
clipper.fit(X)
X_clipped = clipper.transform(X)

print("Original data:")
print(X)
print()
print("Clipped data (range [0.0, 1.0]):")
print(X_clipped)
print()
print("Parameters:", clipper.get_params())

Original data:
[[ 0.5 -0.3]
 [ 1.5  0.7]
 [ 0.2  2.1]]

Clipped data (range [0.0, 1.0]):
[[0.5 0. ]
 [1.  0.7]
 [0.2 1. ]]

Parameters: {'lower': 0.0, 'upper': 1.0}


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import numpy as np

# Generate a synthetic binary classification dataset
X, y = make_classification(n_samples=200, n_features=5,
                           n_informative=3, n_redundant=1,
                           random_state=42)

# Split into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Create a pipeline: scale -> classify
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

# Fit the entire pipeline on training data
pipe.fit(X_train, y_train)

# Evaluate on test data
train_score = pipe.score(X_train, y_train)
test_score = pipe.score(X_test, y_test)
print(f"Training accuracy: {train_score:.4f}")
print(f"Test accuracy:     {test_score:.4f}")
print(f"Pipeline steps:    {[name for name, _ in pipe.steps]}")

Training accuracy: 0.9267
Test accuracy:     0.8800
Pipeline steps:    ['scaler', 'classifier']


In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

# Example data
X = np.array([[1], [2], [3], [4], [5]])  # Feature matrix
y = np.array([1, 2, 3, 3.5, 5])          # Target values

# Create and fit the model
model = LinearRegression()
model.fit(X, y)

# Access learned coefficients (slope)
print("Coefficients (w):", model.coef_)

# Access y-intercept (bias)
print("Intercept (b):   ", model.intercept_)

# Evaluate: R-squared on training data
r2 = model.score(X, y)
print(f"R-squared:         {r2:.6f}")

# Manual verification: predict y for X = 3
y_hat = model.coef_[0] * 3 + model.intercept_
print(f"\nManual prediction for X=3: {model.coef_[0]:.2f} * 3 + {model.intercept_:.2f} = {y_hat:.2f}")
print(f"Actual y for X=3: {y[2]}")

Coefficients (w): [0.95]
Intercept (b):    0.04999999999999938
R-squared:         0.980978

Manual prediction for X=3: 0.95 * 3 + 0.05 = 2.90
Actual y for X=3: 3.0


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create a RandomForestClassifier model
model = RandomForestClassifier()

# Set specific hyperparameters before training
model.set_params(n_estimators=100, max_depth=10, random_state=42)

# Retrieve all current parameters
params = model.get_params()
print("Full parameter dictionary:")
for key, value in sorted(params.items()):
    print(f"  {key}: {value}")

Full parameter dictionary:
  bootstrap: True
  ccp_alpha: 0.0
  class_weight: None
  criterion: gini
  max_depth: 10
  max_features: sqrt
  max_leaf_nodes: None
  max_samples: None
  min_impurity_decrease: 0.0
  min_samples_leaf: 1
  min_samples_split: 2
  min_weight_fraction_leaf: 0.0
  monotonic_cst: None
  n_estimators: 100
  n_jobs: None
  oob_score: False
  random_state: 42
  verbose: 0
  warm_start: False


In [ ]:
from sklearn.linear_model import LogisticRegression

# Inspect estimator tags
lr = LogisticRegression()
tags = lr.__sklearn_tags__()

print("Estimator type:", type(lr).__name__)
print()
print("Key tags:")
print(f"  Supports multiclass:     True (via OvR or multinomial)")
print(f"  Requires fit:            True")
print(f"  Array API support:       {tags.array_api_support}")
print(f"  Non-deterministic:       {tags.non_deterministic}")

Estimator type: LogisticRegression

Key tags:
  Supports multiclass:     True (via OvR or multinomial)
  Requires fit:            True
  Array API support:       False
  Non-deterministic:       False
